# CAPE-TTA V5 — 多初始状态候选回放

**操作：代码执行程序 → 更改运行时类型 → T4 GPU → 保存，然后“全部运行”。**

本阶段检查“不同的候选动作是否真的改善短期进展”，不是只看动作是否不同。
固定 Spatial/task 0，初始状态 0、1、2；4 组采样设置全部保留。
每组 4 个采样动作加 1 个贪心基线（名义候选总数 5），每条分支最多 12 步，
第一动作后使用冻结贪心策略。与 V3 的总 K=4 预算不同。

- 完全重复的执行动作仅评估一次，保留全部采样记录。
- 每个初始状态额外重复基线；一致性不通过就标记失败。
- 每完成一个分支就保存数据；日志持续输出时不要重复点击运行。
- 自动检测和准备环境；若 Colab 分配新运行时，需重新安装并下载模型。
- 结束后自动下载 ZIP；成功或失败都上传回对话。
- **这不是完整回合主实验，也没有参数更新，不能直接当作 CAPE-TTA 效果证明。**
- 固定源代码：4843d93d7616c50e4c4f44889f260c5ac4775bff。代码通过 CPU 测试，本次 GPU 实验尚未执行。


In [ ]:
# V5 paired rollout feasibility: one cell, automatic environment setup and ZIP download.
import datetime as dt
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import traceback
import uuid

SOURCE_SHA = "4843d93d7616c50e4c4f44889f260c5ac4775bff"
SOURCE = Path("/content/cape-source") / SOURCE_SHA
ROOT = Path("/content/cape-feasibility") / ("feasibility-" + dt.datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:8])
ROOT.mkdir(parents=True, exist_ok=False)
PY = Path("/content/cape-env-v3/bin/python")
LIBERO = Path("/content/LIBERO-8f1084e3132a")
state = {"status": "running", "scientific_result": False, "source_sha": SOURCE_SHA}
log = (ROOT / "launcher.log").open("w", encoding="utf-8")

def run(command, env=None):
    line = "$ " + " ".join(map(str, command))
    print(line, flush=True)
    log.write(line + "\n"); log.flush()
    with subprocess.Popen(list(map(str, command)), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1, env=env) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line); log.flush()
        status = proc.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)

try:
    smi = shutil.which("nvidia-smi")
    if smi is None:
        raise RuntimeError("请选择：代码执行程序 → 更改运行时类型 → T4 GPU → 保存，再运行。")
    run([smi, "--query-gpu=name,memory.total", "--format=csv,noheader"])
    if not SOURCE.exists():
        SOURCE.mkdir(parents=True)
        run(["git", "-C", SOURCE, "init"])
        run(["git", "-C", SOURCE, "remote", "add", "origin", "https://github.com/weichezhang/CAPE-TTA.git"])
        run(["git", "-C", SOURCE, "fetch", "--depth", "1", "origin", SOURCE_SHA])
        run(["git", "-C", SOURCE, "checkout", "--detach", SOURCE_SHA])
    actual = subprocess.check_output(["git", "-C", str(SOURCE), "rev-parse", "HEAD"], text=True).strip()
    if actual != SOURCE_SHA:
        raise RuntimeError("Unexpected source revision: " + actual)
    subprocess.run(["git", "-C", str(SOURCE), "diff", "--quiet", "HEAD"], check=True)
    warm = False
    if PY.is_file() and LIBERO.is_dir():
        check = subprocess.run([str(PY), "-c", "import torch, transformers, bitsandbytes, mujoco; assert torch.cuda.is_available(); assert mujoco.__version__ == '2.3.7'; assert torch.__version__.startswith('2.2.0')"],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        warm = check.returncode == 0
        log.write(check.stdout); log.flush()
    if not warm:
        print("准备运行环境；本步骤不会重复 V3 分支实验。", flush=True)
        run([sys.executable, "-u", SOURCE / "scripts/colab_bootstrap.py", "--prepare-only", "--run-dir", ROOT / "setup"])
    else:
        print("检测到可复用的 V3 环境。", flush=True)
    # Install the pinned diagnostic code instead of reusing the old editable package.
    run([PY, "-m", "pip", "install", "--no-build-isolation", "--no-deps", "-e", SOURCE])
    env = dict(os.environ, HF_HOME="/content/cape-hf-cache", PYTHONUNBUFFERED="1",
               MUJOCO_GL="osmesa", PYOPENGL_PLATFORM="osmesa", TF_CPP_MIN_LOG_LEVEL="2")
    run([PY, "-u", SOURCE / "scripts/run_counterfactual_feasibility.py",
         "--libero-root", LIBERO, "--output-root", ROOT / "rollouts", "--init-state-ids", "0", "1", "2", "--horizon", "12"], env=env)
    state["status"] = "feasibility_complete"
    print("短回放实验完成；请上传 ZIP，尚不能据此认定完整任务成功率提升。", flush=True)
except BaseException as exc:
    state["status"] = "failed"
    state["error"] = {"type": type(exc).__name__, "message": str(exc), "traceback": traceback.format_exc()}
    print(traceback.format_exc(), flush=True)
    raise
finally:
    (ROOT / "launcher_status.json").write_text(json.dumps(state, indent=2), encoding="utf-8")
    log.close()
    archive = shutil.make_archive(str(ROOT), "zip", ROOT)
    print("\n请把这个 ZIP 上传回对话：", archive, flush=True)
    from google.colab import files
    files.download(archive)
